## Start a SparkSession

In [23]:
from pyspark.sql import SparkSession
from urllib.request import urlretrieve

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("adsP1")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.driver.memory", "4G")
    .config("spark.executor.memory", "4G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

## Import package

In [24]:
# import package
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.functions import col
from pyspark.sql.functions import count
from pyspark.sql.functions import format_number
from pyspark.sql.functions import unix_timestamp
from pyspark.sql.functions import date_trunc
from pyspark.sql.functions import round
from pyspark.sql.functions import mean, stddev, sqrt
from pyspark.sql.functions import date_format
from pyspark.sql.functions import avg
import math

# 1."A" represents pre-process major dataset / 2. "B" represnts pre-process external dataset / 3. "C" represents pre-process for combinin major dataset & external dataset

## A. Load Taxi Dataset

In [25]:
# read 2022 9&10&11 months months yellow_taxi 
df_2022_yellow = spark.read.parquet('/Users/tianhao/Desktop/adsP1/data/raw/tlc_data/2022/yellow_taxi_*.parquet')

# read 2023 9&10&11 months months yellow_taxi 
df_2023_yellow = spark.read.parquet('/Users/tianhao/Desktop/adsP1/data/raw/tlc_data/2023/yellow_taxi_*.parquet')


# view counts
print(f'Total counts of 2022 Month 9-11 Yellow Taxi {df_2022_yellow.count()} Records')
print(f'Total counts of 2023 Month 9-11 Yellow Taxi {df_2023_yellow.count()} Records')

Total counts of 2022 Month 9-11 Yellow Taxi 10111895 Records
Total counts of 2023 Month 9-11 Yellow Taxi 9708722 Records


## A. Inspect the attribute information of Yellow Taxi

In [26]:
df_2023_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



## A. Step 1 : delete the irrelvant variables

In [27]:
# since our aim is to use predict the 
# in the feature engineering part aggreate total_amount and tips
df_2022_yellow = df_2022_yellow.drop('store_and_fwd_flag','Fare_amount', 'Extra', \
                                     'MTA_tax', 'Improvement_surcharge', 'Tolls_amount', \
                                    'Congestion_Surcharge', 'Airport_fee','payment_type','VendorID')
df_2023_yellow = df_2023_yellow.drop('store_and_fwd_flag','Fare_amount', 'Extra', \
                                     'MTA_tax', 'Improvement_surcharge', 'Tolls_amount', \
                                    'Congestion_Surcharge', 'Airport_fee','payment_type','VendorID')

## A. Step 2 : handle the missing value

### (1). Check missing proportion in 2022 and 2023 New York Yellow_taxi month 9-11

In [28]:
print(f'Total counts of 2022 Month 9-12 Yellow Taxi {df_2022_yellow.count()} Records')

yellow_missing_proportion_2 = (
    df_2022_yellow.agg(*
        [(1 - (F.count(c) / F.count("*"))).alias(c) for c in df_2023_yellow.columns]
        ).toPandas().T.reset_index()
            )
yellow_missing_proportion_2.columns = ['Attribute', 'Missing_Proportion(%)']
yellow_missing_proportion_2['Missing_Proportion(%)'] = \
(yellow_missing_proportion_2['Missing_Proportion(%)'] * 100).round(3)

print(yellow_missing_proportion_2.to_string(index=False))

Total counts of 2022 Month 9-12 Yellow Taxi 10111895 Records
            Attribute  Missing_Proportion(%)
 tpep_pickup_datetime                  0.000
tpep_dropoff_datetime                  0.000
      passenger_count                  3.675
        trip_distance                  0.000
           RatecodeID                  3.675
         PULocationID                  0.000
         DOLocationID                  0.000
           tip_amount                  0.000
         total_amount                  0.000


In [29]:
print(f'Total counts of 2023 Month 9-11 Yellow Taxi {df_2023_yellow.count()} Records')

yellow_missing_proportion_2023 = (
    df_2023_yellow.agg(*
        [(1 - (F.count(c) / F.count("*"))).alias(c) for c in df_2023_yellow.columns]
        ).toPandas().T.reset_index()
            )

yellow_missing_proportion_2023.columns = ['Attribute', 'Missing_Proportion(%)']

yellow_missing_proportion_2023['Missing_Proportion(%)'] = \
    (yellow_missing_proportion_2023['Missing_Proportion(%)'] * 100).round(3)

print(yellow_missing_proportion_2023.to_string(index=False))
# have a check on description for Rate/store_fwd_flag/enhail_fee

Total counts of 2023 Month 9-11 Yellow Taxi 9708722 Records
            Attribute  Missing_Proportion(%)
 tpep_pickup_datetime                  0.000
tpep_dropoff_datetime                  0.000
      passenger_count                  4.407
        trip_distance                  0.000
           RatecodeID                  4.407
         PULocationID                  0.000
         DOLocationID                  0.000
           tip_amount                  0.000
         total_amount                  0.000


### (2). Remove the missing value in Passenger & RatecodeID

In [30]:
""" Since the missing the highest proportion column in 2022 and 2023 are respectively only 3.675% and 4.407%,
 and remain the previous feature of data and reduce future model bias
 it is reasonable to directly delete the missing value. 
"""

df_2022_yellow = df_2022_yellow.dropna(subset=['passenger_count', 'RatecodeID'])

df_2023_yellow = df_2023_yellow.dropna(subset=['passenger_count', 'RatecodeID'])

# check the missing count
missing_values_2022 = df_2022_yellow.\
    select([sum(col(c).isNull().cast("int")).\
     alias(c) for c in df_2022_yellow.columns])

missing_values_2023 = df_2023_yellow.\
    select([sum(col(c).isNull().cast("int")).\
     alias(c) for c in df_2023_yellow.columns])

print(f'After removing missing values, counts of 2022 Month 9-11 Yellow Taxi {df_2022_yellow.count()} Records')

print(f'After removing missing values, counts of 2023 Month 9-11 Yellow Taxi {df_2023_yellow.count()} Records')

# Check the missing counts
print(missing_values_2022.show(vertical=True))
print(missing_values_2023.show(vertical=True))

After removing missing values, counts of 2022 Month 9-11 Yellow Taxi 9740324 Records
After removing missing values, counts of 2023 Month 9-11 Yellow Taxi 9280893 Records
-RECORD 0--------------------
 tpep_pickup_datetime  | 0   
 tpep_dropoff_datetime | 0   
 passenger_count       | 0   
 trip_distance         | 0   
 RatecodeID            | 0   
 PULocationID          | 0   
 DOLocationID          | 0   
 tip_amount            | 0   
 total_amount          | 0   

None
-RECORD 0--------------------
 tpep_pickup_datetime  | 0   
 tpep_dropoff_datetime | 0   
 passenger_count       | 0   
 trip_distance         | 0   
 RatecodeID            | 0   
 PULocationID          | 0   
 DOLocationID          | 0   
 tip_amount            | 0   
 total_amount          | 0   

None


## A. Step 3: Data type transformation & Duplicate removal

In [31]:
# 1. Uniform the datatype for preparing feature engineering
df_2022_yellow = df_2022_yellow.withColumn("passenger_count", col("passenger_count").cast("integer"))
df_2022_yellow = df_2022_yellow.withColumn("RatecodeID", col("RatecodeID").cast("integer"))
df_2022_yellow = df_2022_yellow.withColumn("tpep_pickup_datetime", col("tpep_pickup_datetime").cast("timestamp"))
df_2022_yellow = df_2022_yellow.withColumn("tpep_dropoff_datetime", col("tpep_dropoff_datetime").cast("timestamp"))

df_2023_yellow = df_2023_yellow.withColumn("passenger_count", col("passenger_count").cast("integer"))
df_2023_yellow = df_2023_yellow.withColumn("RatecodeID", col("RatecodeID").cast("integer"))
df_2023_yellow = df_2023_yellow.withColumn("tpep_pickup_datetime", col("tpep_pickup_datetime").cast("timestamp"))
df_2023_yellow = df_2023_yellow.withColumn("tpep_dropoff_datetime", col("tpep_dropoff_datetime").cast("timestamp"))

# 2. Remove the duplicates
print(df_2022_yellow.count())
print(df_2023_yellow.count())

df_2022_yellow.dropDuplicates()
df_2023_yellow.dropDuplicates()

print(df_2022_yellow.count())
print(df_2023_yellow.count())

9740324
9280893
9740324
9280893


## A. Step 4 : Feature Engineering part 1 

### (1). Aggreated Total_revenue for one order

In [32]:
"""
In order to obtain more accurate income predictions in the model, I added the total amount 
(excluding tips according to the data dictionary) to tip amount to create a new aggregated field called total amount. 
This ensures that all relevant revenue components are taken into account, improving the model's ability to predict total revenue
and then drop total_amount and tip_amount
"""

df_2022_yellow = df_2022_yellow.withColumn(
    "total_revenue",
    round(col("total_amount") + col("tip_amount"), 2)
)

df_2023_yellow = df_2023_yellow.withColumn(
    "total_revenue",
    round(col("total_amount") + col("tip_amount"), 2)
)

df_2022_yellow = df_2022_yellow.drop('total_amount', 'tip_amount')

df_2023_yellow = df_2023_yellow.drop('total_amount', 'tip_amount')

### (2). trip_duration = drop_off_time - pick_up_time (Mintues)

In [33]:
df_2022_yellow = df_2022_yellow.withColumn(
    "duration_minutes",
    round((unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60, 1)
)

df_2023_yellow = df_2023_yellow.withColumn(
    "duration_minutes",
    round((unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60, 1)
)

In [34]:
df_2022_yellow.show(5)

+--------------------+---------------------+---------------+-------------+----------+------------+------------+-------------+----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|PULocationID|DOLocationID|total_revenue|duration_minutes|
+--------------------+---------------------+---------------+-------------+----------+------------+------------+-------------+----------------+
| 2022-09-01 00:28:12|  2022-09-01 00:36:22|              1|          2.1|         1|         100|         239|         16.4|             8.2|
| 2022-09-01 00:51:58|  2022-09-01 01:14:43|              1|          8.7|         1|         161|         243|         31.3|            22.8|
| 2022-09-01 00:08:29|  2022-09-01 00:26:29|              1|          8.3|         1|         138|         233|        39.35|            18.0|
| 2022-09-01 00:02:24|  2022-09-01 00:09:39|              1|         1.32|         1|         238|         166|          8.8|             7.3|

In [35]:
df_2022_yellow.printSchema()
df_2023_yellow.printSchema()

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- duration_minutes: double (nullable = true)

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- duration_minutes: double (nullable = true)



## B.  Preprocess the External NYC Hourly Weather Dataset(2022 & 2023: 9-11 Month)

## B. Step 1 : Load data & Remain Relevant Features & Rename column

In [36]:
# Load the external dataset
df_external_2022 = spark.read.csv("/Users/tianhao/Desktop/adsP1/data/raw/external/NYC_2022_autumn_weather.csv",\
                                       header=True, inferSchema=True)

df_external_2023 = spark.read.csv("/Users/tianhao/Desktop/adsP1/data/raw/external/NYC_2023_autumn_weather.csv",\
                                       header=True, inferSchema=True)

# remain Feels Like / Precipitation / Wind Speed / Visibility as our feature

df_external_2022 = df_external_2022.select("datetime","feelslike","precip","windspeed","visibility")
df_external_2023 = df_external_2023.select("datetime","feelslike","precip","windspeed","visibility")


# Rename the column for better understanding of the attribute
df_external_2022 = df_external_2022 \
    .withColumnRenamed("feelslike", "feels_like") \
    .withColumnRenamed("precip", "precipitation") \
    .withColumnRenamed("windspeed", "wind_speed")

df_external_2023 = df_external_2023 \
    .withColumnRenamed("feelslike", "feels_like") \
    .withColumnRenamed("precip", "precipitation") \
    .withColumnRenamed("windspeed", "wind_speed")

df_external_2022.dropDuplicates()
df_external_2023.dropDuplicates()

# view the missing counts
df_external_2022.select([F.sum(F.col(c).isNull().cast("int")). \
                             alias(c) for c in df_external_2022.columns]).show(vertical=True)

df_external_2023.select([F.sum(F.col(c).isNull().cast("int")). \
                             alias(c) for c in df_external_2023.columns]).show(vertical=True)


-RECORD 0------------
 datetime      | 0   
 feels_like    | 0   
 precipitation | 0   
 wind_speed    | 0   
 visibility    | 0   

-RECORD 0------------
 datetime      | 0   
 feels_like    | 0   
 precipitation | 0   
 wind_speed    | 0   
 visibility    | 0   



In [37]:
external_decription = {
    "Attribute": ["feels_like", "precipitation",  "wind_speed", "visibility"],
    "Unit (US)": ["F", "inches", "mph", "miles"]
}

external_decription = pd.DataFrame(external_decription)

external_decription


,Attribute,Unit (US)
0,feels_like,F
1,precipitation,inches
2,wind_speed,mph
3,visibility,miles


In [38]:
# In case there was a date storage error when merging data sets that resulted in null values

df_external_2022 = df_external_2022.filter(
    (col("datetime") >= "2022-09-01 00:00:00") & 
    (col("datetime") <= "2022-11-30 23:59:59")
)

df_external_2023 = df_external_2023.filter(
    (col("datetime") >= "2023-09-01 00:00:00") & 
    (col("datetime") <= "2023-11-30 23:59:59")
)

df_2022_yellow = df_2022_yellow.filter(
    (col("tpep_pickup_datetime") >= "2022-09-01 00:00:00") & 
    (col("tpep_pickup_datetime") <= "2022-11-30 23:59:59")
)

df_2023_yellow = df_2023_yellow.filter(
    (col("tpep_pickup_datetime") >= "2023-09-01 00:00:00") & 
    (col("tpep_pickup_datetime") <= "2023-11-30 23:59:59")
)

## C. Step 1 : Combine External dataset & Main dataset

In [39]:
# Truncate the datetime column of df 2023 yellow to hour

df_2022_yellow = df_2022_yellow.withColumn("datetime_hour", date_trunc("hour", col("tpep_pickup_datetime")))
df_2023_yellow = df_2023_yellow.withColumn("datetime_hour", date_trunc("hour", col("tpep_pickup_datetime")))
print(df_2022_yellow.count())
print(df_2023_yellow.count())
df_external_2022 = df_external_2022.dropDuplicates(["datetime"])
df_external_2023 = df_external_2023.dropDuplicates(["datetime"])

# The connection is based on the datetime columns of datetime_hour and df_external_2023
df_combined_2022 = df_2022_yellow.join(
    df_external_2022,
    df_2022_yellow["datetime_hour"] == df_external_2022["datetime"],
    "inner"
)

df_combined_2023 = df_2023_yellow.join(
    df_external_2023,
    df_2023_yellow["datetime_hour"] == df_external_2023["datetime"],
    "inner"
)

df_combined_2022 = df_combined_2022.drop("datetime_hour", "datetime")
df_combined_2023 = df_combined_2023.drop("datetime_hour", "datetime")

print(f"After combination, the main dataset 2022 month 9-11 has {df_combined_2022.count()} Records.")
print(f"After combination, the main dataset 2023 month 9-11 has {df_combined_2023.count()} Records.")

9740199
9280861


After combination, the main dataset 2022 month 9-11 has 9740199 Records.


After combination, the main dataset 2023 month 9-11 has 9280861 Records.


In [40]:
df_combined_2022.printSchema()

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- feels_like: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- visibility: double (nullable = true)



## C. Step 2 : Handling with outliers

### Outlier 1. filter trip_duration > 6 hours and trip_duration < 2 minutes

In [41]:
df_combined_2022 = df_combined_2022.filter((col("duration_minutes") > 2) & (col("duration_minutes") <= 360))
df_combined_2023 = df_combined_2023.filter((col("duration_minutes") > 2) & (col("duration_minutes") <= 360))

### Outlier 2. Remove the passengers records are lower than 6 and higher than 1 

In [42]:
# outlier 1. remove the passengers records are lower than 6 and higher than 1 
df_combined_2022 = df_combined_2022.filter(
    (col("passenger_count") >= 1) & (col("passenger_count") <= 6)
)
df_combined_2023 = df_combined_2023.filter(
    (col("passenger_count") >= 1) & (col("passenger_count") <= 6)
)

### Outlier 3: filter Trip_distance using Z-score and filter Trip_distance < 0.3 miles

In [43]:
# 1. For 2022 trip_distance outlier removal

## Firstly, we use filter trip_distance < 0.3 miles
df_combined_2022 = df_combined_2022.filter(col("trip_distance") > 0.3)
## Then use the z-score
N_2022 = df_combined_2022.count()
threshold1 = math.sqrt(2 * math.log(N_2022))
## Calculate the z-fraction of the trip distance
mean_trip_distance_N_2022 = df_combined_2022.select(mean(col("trip_distance"))).first()[0]
stddev_trip_distance_N_2022 = df_combined_2022.select(stddev(col("trip_distance"))).first()[0]

df_combined_2022 = df_combined_2022.withColumn(
    "z_score_trip_distance",
    (col("trip_distance") - mean_trip_distance_N_2022) / stddev_trip_distance_N_2022
)

df_combined_2022 = df_combined_2022.filter(
    abs(col("z_score_trip_distance")) <= threshold1
).drop("z_score_trip_distance")


# 2. For 2023 trip_distance outlier removal
## filter trip_distance < 0.3 miles
df_combined_2023 = df_combined_2023.filter(col("trip_distance") > 0.2)
## Then use the z-score
N_2023 = df_combined_2023.count()
threshold2 = math.sqrt(2 * math.log(N_2023))
## Calculate the z-fraction of the trip distance
mean_trip_distance_N_2023 = df_combined_2023.select(mean(col("trip_distance"))).first()[0]
stddev_trip_distance_N_2023 = df_combined_2023.select(stddev(col("trip_distance"))).first()[0]

df_combined_2023 = df_combined_2023.withColumn(
    "z_score_trip_distance",
    (col("trip_distance") - mean_trip_distance_N_2023) / stddev_trip_distance_N_2023
)

df_combined_2023 = df_combined_2023.filter(
    abs(col("z_score_trip_distance")) <= threshold2
).drop("z_score_trip_distance")

Exception in thread "serve-DataFrame" java.net.SocketTimeoutException: Accept timed out
	at java.base/java.net.PlainSocketImpl.socketAccept(Native Method)
	at java.base/java.net.AbstractPlainSocketImpl.accept(AbstractPlainSocketImpl.java:474)
	at java.base/java.net.ServerSocket.implAccept(ServerSocket.java:565)
	at java.base/java.net.ServerSocket.accept(ServerSocket.java:533)
	at org.apache.spark.security.SocketAuthServer$$anon$1.run(SocketAuthServer.scala:65)


### Outlier 4 : Total revenue (same method in handling trip_distance)

In [44]:
# Firstly, according to initial fee and surcharge from yellow taxi infomation
# we filter the total_revenue >= 4
N_2022 = df_combined_2022.count()
threshold1 = math.sqrt(2 * math.log(N_2022))
df_combined_2022 = df_combined_2022.filter(col("total_revenue") >= 4)
## Calculate the z-fraction of the total_revenue for 2022
mean_total_revenue_N_2022 = df_combined_2022.select(mean(col("total_revenue"))).first()[0]
stddev_total_revenue_N_2022 = df_combined_2022.select(stddev(col("total_revenue"))).first()[0]

df_combined_2022 = df_combined_2022.withColumn(
    "z_score_total_revenue",
    (col("total_revenue") - mean_total_revenue_N_2022) / stddev_total_revenue_N_2022
)

## use the Z-score to filter outlier

df_combined_2022 = df_combined_2022.filter(
    abs(col("z_score_total_revenue")) <= threshold1
).drop("z_score_total_revenue")


# filter the total_revenue >= 4
N_2023 = df_combined_2023.count()
threshold2 = math.sqrt(2 * math.log(N_2023))
df_combined_2023 = df_combined_2023.filter(col("total_revenue") >= 4)
## Calculate the z-fraction of the total_revenue for 2023
mean_total_revenue_N_2023 = df_combined_2023.select(mean(col("total_revenue"))).first()[0]
stddev_total_revenue_N_2023 = df_combined_2023.select(stddev(col("total_revenue"))).first()[0]
df_combined_2023 = df_combined_2023.withColumn(
    "z_score_total_revenue",
    (col("total_revenue") - mean_total_revenue_N_2023) / stddev_total_revenue_N_2023
)

df_combined_2023 = df_combined_2023.filter(
    abs(col("z_score_total_revenue")) <= threshold2
).drop("z_score_total_revenue")

In [45]:
df_combined_2022.describe()

24/08/24 23:47:42 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


summary,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,total_revenue,duration_minutes,feels_like,precipitation,wind_speed,visibility
count,9187855,9187855,9187855,9187855,9187855,9187855,9187855,9187855,9187855,9187855,9187855
mean,1.414532880634272,3.6625613693294765,1.3526159261329223,165.5693718501217,163.07788085467175,24.840597271767397,17.012129392552172,59.893170462545875,0.004236760375517189,6.71059606402222,9.343720530658697
stddev,0.9248972913650828,4.498994714264473,5.331030699428307,64.65920204959468,69.93173410725568,18.422235208696687,13.336132963064035,12.264618868101783,0.024397319980564675,3.8985894145821156,1.7039279187391645
min,1,0.31,1,1,1,4.05,2.1,20.9,0.0,0.0,0.3
max,6,312.07,99,265,265,138.05,359.4,90.3,0.588,26.4,9.9


In [46]:
df_combined_2023.describe()

summary,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,total_revenue,duration_minutes,feels_like,precipitation,wind_speed,visibility
count,8821775,8821775,8821775,8821775,8821775,8821775,8821775,8821775,8821775,8821775,8821775
mean,1.3771768153234467,3.54094423854599,1.6639933573458856,165.61502305375052,164.78038660020235,33.59545297176523,17.72551925207783,61.25972518001662,4.965130033353238E-4,6.32588118603476,9.541703852195056
stddev,0.8656915683238143,4.6129801963676105,7.658660700008565,63.43416669329241,69.60697201945786,26.257490190334728,14.282049826841833,13.840639143084159,0.004738683776499787,3.7637228347566243,1.1886258489945096
min,1,0.21,1,1,1,4.0,2.1,19.2,0.0,0.0,1.2
max,6,342.1,99,265,265,507.05,360.0,100.1,0.183,24.8,9.9


## C. Step 3 : Feature Engineering: Assign DAY OF week & Hour to each column for EDA

In [47]:
# Look at the pattern by day of week or hour in EDA session
df_combined_2022 = df_combined_2022.withColumn("day_of_week", date_format(col("tpep_pickup_datetime"), "EEEE"))
df_combined_2023 = df_combined_2023.withColumn("day_of_week", date_format(col("tpep_pickup_datetime"), "EEEE"))
df_combined_2022 = df_combined_2022.withColumn("hour", hour(col("tpep_pickup_datetime")))
df_combined_2023 = df_combined_2023.withColumn("hour", hour(col("tpep_pickup_datetime")))

## Splicing Yellow taxi data for 2022 and 2023

In [48]:
print(df_combined_2022.count())
print(df_combined_2023.count())
df_all = df_combined_2022.union(df_combined_2023)
print(df_all.count())

9187855


8821775


18009630


In [49]:
df_all.printSchema()

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- feels_like: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- visibility: double (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- hour: integer (nullable = true)



## C. Step 3 : Feature Engineering : Generate peak hour weather features

In [50]:
# Step 1: Define peak hours 7-9 am and 17-19 pm
df_all = df_all.withColumn(
    "is_peak_hour",
    when((col("hour").between(7, 9)) | (col("hour").between(17, 19)), 1).otherwise(0)
)
# Filter out the data during peak hours. If is_peak_hour = 1, the peak and other values are 0
df_peak_hours = df_all.filter(col("is_peak_hour") == 1)

In [51]:
# Aggregate peak period data to calculate the average of weather characteristics and peak period order volumes

df_peak_weather = df_peak_hours.groupBy("day_of_week").agg(
    avg(col("feels_like")).alias("avg_feels_like_peak"),
    avg(col("precipitation")).alias("avg_precipitation_peak"),
    avg(col("wind_speed")).alias("avg_wind_speed_peak"),
    avg(col("visibility")).alias("avg_visibility_peak"),
    avg(col("duration_minutes")).alias("avg_duration_minutes_peak"),
    avg(col("trip_distance")).alias("avg_trip_distance_peak"),
    count(col("*")).alias("Peak_Trip_Count")  # 计算高峰时段的订单量
)

In [52]:
# Add peak hour weather features and totals back to the original dataset
df_all = df_all.join(
    df_peak_weather,
    on="day_of_week",
    how="left"
)

In [53]:
df_all.printSchema()

root
 |-- day_of_week: string (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- feels_like: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- visibility: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- is_peak_hour: integer (nullable = false)
 |-- avg_feels_like_peak: double (nullable = true)
 |-- avg_precipitation_peak: double (nullable = true)
 |-- avg_wind_speed_peak: double (nullable = true)
 |-- avg_visibility_peak: double (nullable = true)
 |-- avg_duration_minutes_peak: double (nullable = true)
 |-- avg_t

## Save the Data

In [54]:
df_combined_2022.coalesce(1).write.mode("overwrite").parquet("/Users/tianhao/Desktop/adsP1/data/curate/tlc_data/first_clean/df_combined_2022.parquet")
df_combined_2023.coalesce(1).write.mode("overwrite").parquet("/Users/tianhao/Desktop/adsP1/data/curate/tlc_data/first_clean/df_combined_2023.parquet")
df_all.coalesce(1).write.mode("overwrite").parquet("/Users/tianhao/Desktop/adsP1/data/curate/tlc_data/first_clean/df_all.parquet")
